# Cloud Architecture & Deployment — Reference Patterns — Hands-On

**AI Architecture · Week 20a**

Offline notebook: platform landscape, Dockerfile discipline, manifest validation, canary evaluation, and a realistic GitHub Actions flow. No network calls.

## 0. Platform and deployment overview

```mermaid
flowchart LR
  GHA[GitHub Actions] --> ACR[Azure Container Registry]
  ACR --> ACA[Azure Container Apps RAG API]
  ACR --> AKS[AKS GPU Workers]
  ACA --> AOAI[Azure OpenAI Private Endpoint]
  ACA --> PG[Postgres pgvector]
  ACA --> KV[Key Vault]
  ACA --> AI[Application Insights]
```

Compare Azure AI Foundry / Azure OpenAI, AWS Bedrock, and Google Vertex AI on model catalog, regions, private networking, data residency, quota, provisioned capacity, content filters, and key management.

## 1. Imports

In [ ]:
import re
from dataclasses import dataclass
from enum import Enum
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator, model_validator

## 2. Dockerfile discipline as a reviewable artifact

In [ ]:
dockerfile = r'''
FROM python:3.11-slim-bookworm AS build
WORKDIR /app
COPY requirements.lock .
RUN pip wheel --no-cache-dir --wheel-dir /wheels -r requirements.lock
FROM python:3.11-slim-bookworm AS runtime
RUN useradd --create-home --uid 10001 appuser
WORKDIR /app
COPY --from=build /wheels /wheels
RUN pip install --no-cache-dir /wheels/* && rm -rf /wheels
COPY --chown=appuser:appuser src/ ./src/
USER appuser
CMD ["python", "-m", "uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''
checks = {'multi_stage': 'AS build' in dockerfile and 'AS runtime' in dockerfile, 'non_root': 'USER appuser' in dockerfile, 'no_secret_env': 'AZURE_OPENAI_KEY=' not in dockerfile and 'SECRET=' not in dockerfile, 'cache_hygiene': '--no-cache-dir' in dockerfile}
print(checks)

## 3. Manifest validator

In [ ]:
MEMORY_RE = re.compile(r'^(128|256|512)Mi$|^[1-9][0-9]*Gi$')
class Probe(BaseModel):
    model_config = ConfigDict(extra='forbid')
    path: str
    initial_delay_seconds: int = Field(default=5, ge=0, le=120)
    period_seconds: int = Field(default=10, ge=1, le=60)
    @field_validator('path')
    @classmethod
    def valid_path(cls, value):
        if not value.startswith('/') or ' ' in value or '?' in value:
            raise ValueError('probe path must be absolute and simple')
        return value
class ScaleRule(BaseModel):
    model_config = ConfigDict(extra='forbid')
    name: str
    type: str
    metadata: dict[str, str] = Field(default_factory=dict)
class DeploymentManifest(BaseModel):
    model_config = ConfigDict(extra='forbid')
    name: str = Field(pattern=r'^[a-z][a-z0-9-]{2,40}$')
    image: str; tag: str
    cpu: float = Field(gt=0, le=4)
    memory: str
    replicas: int = Field(ge=0, le=50)
    min_replicas: int = Field(alias='minReplicas', ge=0, le=50)
    max_replicas: int = Field(alias='maxReplicas', ge=1, le=100)
    ingress: bool = True
    env: dict[str, str] = Field(default_factory=dict)
    secret_refs: dict[str, str] = Field(default_factory=dict, alias='secretRefs')
    liveness: Probe; readiness: Probe
    scale_rules: list[ScaleRule] = Field(default_factory=list, alias='scaleRules')
    @field_validator('memory')
    @classmethod
    def memory_shape(cls, value):
        if not MEMORY_RE.match(value): raise ValueError('memory must look like 512Mi or 2Gi')
        return value
    @model_validator(mode='after')
    def cross_checks(self):
        if self.min_replicas > self.max_replicas: raise ValueError('minReplicas must be <= maxReplicas')
        if self.replicas and not (self.min_replicas <= self.replicas <= self.max_replicas): raise ValueError('replicas must be between minReplicas and maxReplicas')
        for key, value in self.env.items():
            if any(word in key.upper() for word in ('KEY','SECRET','TOKEN','PASSWORD','CONNECTION_STRING')) or value.lower().startswith(('sk-','eyj','postgres://')):
                raise ValueError(f'{key} belongs in secretRefs, not env')
        for name, ref in self.secret_refs.items():
            if not ref.startswith('keyvault:') or '=' in ref: raise ValueError(f'secretRef {name} must reference keyvault:<secret-name>')
        return self

## 4. Exercise manifest validator

In [ ]:
good = {'name':'rag-api-prod','image':'acr.azurecr.io/rag-api','tag':'2026.07.18.shaabc','cpu':1.0,'memory':'2Gi','replicas':2,'minReplicas':1,'maxReplicas':10,'ingress':True,'env':{'ENVIRONMENT':'prod','PROMPT_VERSION':'rag-v12'},'secretRefs':{'AZURE_OPENAI_KEY':'keyvault:aoai-key','DB_PASSWORD':'keyvault:pg-password'},'liveness':{'path':'/healthz'},'readiness':{'path':'/readyz'},'scaleRules':[{'name':'http','type':'http','metadata':{'concurrentRequests':'50'}}]}
print('accepted:', DeploymentManifest.model_validate(good).name)
for label, patch in [('raw secret in env', {'env': {'AZURE_OPENAI_KEY': 'sk-live-secret'}}), ('min > max', {'minReplicas': 8, 'maxReplicas': 3, 'replicas': 4}), ('bad probe path', {'readiness': {'path': 'ready now'}})]:
    try:
        DeploymentManifest.model_validate({**good, **patch})
    except ValidationError as exc:
        print('rejected', label, '->', exc.errors()[0]['msg'])

## 5. Canary release evaluator

In [ ]:
class Decision(str, Enum):
    PROMOTE = 'PROMOTE'; HOLD = 'HOLD'; ROLLBACK = 'ROLLBACK'
@dataclass(frozen=True)
class Metrics:
    p95_latency_ms: float; error_rate: float; groundedness: float
@dataclass(frozen=True)
class RolloutConfig:
    traffic_percent: int; max_latency_regression_pct: float = 20.0; max_error_rate_abs: float = 0.02; max_error_rate_delta: float = 0.01; min_groundedness: float = 0.86; rollback_groundedness_drop: float = 0.05
def evaluate_canary(baseline, canary, cfg):
    reasons = []
    latency_delta = ((canary.p95_latency_ms - baseline.p95_latency_ms) / baseline.p95_latency_ms) * 100
    error_delta = canary.error_rate - baseline.error_rate
    groundedness_drop = baseline.groundedness - canary.groundedness
    if groundedness_drop >= cfg.rollback_groundedness_drop or canary.groundedness < cfg.min_groundedness - 0.03:
        return Decision.ROLLBACK, [f'groundedness dropped {groundedness_drop:.3f} to {canary.groundedness:.3f}']
    if canary.error_rate > cfg.max_error_rate_abs + 0.02:
        return Decision.ROLLBACK, [f'error rate {canary.error_rate:.3%} is unsafe']
    if latency_delta > cfg.max_latency_regression_pct: reasons.append(f'p95 latency regression {latency_delta:.1f}%')
    if error_delta > cfg.max_error_rate_delta or canary.error_rate > cfg.max_error_rate_abs: reasons.append(f'error-rate delta {error_delta:.3%}')
    if canary.groundedness < cfg.min_groundedness: reasons.append(f'groundedness {canary.groundedness:.3f} below floor')
    return (Decision.HOLD, reasons) if reasons else (Decision.PROMOTE, [f'{cfg.traffic_percent}% canary within guardrails'])

## 6. Canary scenarios

In [ ]:
baseline = Metrics(900, 0.008, 0.91)
config = RolloutConfig(traffic_percent=10)
for name, metrics in {'healthy canary': Metrics(880, 0.007, 0.915), 'latency regression': Metrics(1180, 0.009, 0.905), 'groundedness drop': Metrics(870, 0.007, 0.82)}.items():
    decision, reasons = evaluate_canary(baseline, metrics, config)
    print(name, '->', decision.value, '|', '; '.join(reasons))

## 7. GitHub Actions workflow shape for ACR and Container Apps

In [ ]:
workflow = r'''
name: deploy-ai-services
on: {push: {branches: [main]}}
permissions: {id-token: write, contents: read}
strategy: {matrix: {service: [rag-api, ingestion-worker]}}
jobs:
  build_scan_deploy:
    runs-on: ubuntu-latest
    environment: production
    steps:
      - uses: actions/checkout@v4
      - uses: azure/login@v2
        with:
          client-id: ${{ vars.AZURE_CLIENT_ID }}
          tenant-id: ${{ vars.AZURE_TENANT_ID }}
          subscription-id: ${{ vars.AZURE_SUBSCRIPTION_ID }}
      - name: Build and push immutable image
        run: docker build -t myacr.azurecr.io/${{ matrix.service }}:${{ github.sha }} services/${{ matrix.service }} && docker push myacr.azurecr.io/${{ matrix.service }}:${{ github.sha }}
      - name: Deploy revision to Container Apps
        run: az containerapp update --name ${{ matrix.service }} --resource-group rg-ai-prod --image myacr.azurecr.io/${{ matrix.service }}:${{ github.sha }}
      - name: Smoke and canary gate
        run: python scripts/evaluate_canary.py --image ${{ github.sha }} --prompt rag-v12 --model gpt-4o-prod --index index_20260718
'''
stages = ['actions/checkout', 'azure/login', 'docker build', 'docker push', 'containerapp update', 'evaluate_canary']
print({stage: stage in workflow for stage in stages})

## Exercises
1. Add a KEDA queue-depth scale rule to the manifest and validate required metadata.
2. Extend the canary evaluator with cost/request and refusal-rate thresholds.
3. Write a Bicep or Terraform module boundary for Key Vault, Container Apps, and Azure OpenAI private endpoint.
4. Design a rollback drill for image, prompt, model deployment, vector index, and fine-tuned adapter.

## Links
- Literature note: `02 Literature Notes/AI Architecture/Cloud Architecture & Deployment — Reference Patterns`
- Snippets: `04 Code Snippets/AI Architecture/AI Week 20a Cloud Deployment Manifest Validator`, `.../AI Week 20a Canary Release Evaluator`
- MOC: `06 Maps of Content/AI Architecture Concepts`